*Module 9 of 9*

> **¿Prefieres español?** Abre [`09_del_navegador_a_produccion.ipynb`](../es/09_del_navegador_a_produccion.ipynb) — es el mismo módulo, en español.


# 🚀 Module 9 — From your browser to production

🧭 **Objectives** — connect everything you did in the browser to the real
production tool, **[geocrop_analysis_mx](https://github.com/abxda/geocrop_analysis_mx)**.
Understand what changes at scale, what the browser *cannot* do and why, and
exactly how to run the production pipeline yourself on the same Yaqui Valley
data — offline, no Google Earth Engine needed.

You are now equipped to use that tool *consciously*: you know what each phase
does, what it needs, and how to judge its output.

![full pipeline](../../anim/en/10_full_pipeline.svg)


## Same story, bigger everything

Every step you learned maps one-to-one onto a phase of the production
pipeline — it just runs bigger:

| You did (browser) | `geocrop_analysis_mx` (desktop) | Phase |
|---|---|---|
| 1 tile, 1 month | whole regions, **many months** | `download` |
| geomedian given to you | builds geomedians from open STAC/COG **or** GEE | `download` |
| optical bands only | **+ Sentinel-1 radar (SAR)** and RVI | `download` |
| `shepherd-wasm` (NumPy) | `pyshepseg` (numba-accelerated) | `segment` |
| purity filter, by hand | spatial join + purity filter | `label` |
| mean/std of 13 layers | full zonal stats over **~846 features** | `extract` |
| Random Forest | **TPOT (AutoML)** searches the best pipeline | `train` |
| paint the array | writes a **GeoPackage** for QGIS | `predict` |

📚 **Why radar?** Clouds block optical sensors; **Synthetic Aperture Radar**
(Sentinel-1) sees through them and senses structure and moisture — extra
clues, especially in cloudy seasons. 📚 **Why many months?** That is the
**phenology** (Module 4) — the season-long NDVI story that separates
look-alike crops. Stacked months × bands × indices form a **data cube**.


## Two ways to download the images (free by default, GEE optional)

You never provided a Google Earth Engine account in this course — and the
production pipeline does not need one either. `geocrop_analysis_mx` has
**two download backends**, chosen in the config file:

- **`download_backend: "stac"` (default, free, no account).** Imagery is
  pulled straight from open **STAC/COG** cloud catalogs and the geomedian is
  computed locally. The optical provider is set by `hls_provider`:
  - `"mpc"` — **Microsoft Planetary Computer**, anonymous, no token (the
    default fallback; its HLS archive has gaps before ~2020).
  - `"nasa"` — **NASA LPCLOUD**, the complete authoritative HLS archive;
    needs a *free* NASA Earthdata token in `EARTHDATA_TOKEN` (Profile →
    Generate Token at urs.earthdata.nasa.gov).
  - `"earthsearch"` — **Element 84 / AWS** Sentinel-2, anonymous and the one
    that works inside a browser (CORS-enabled).
  - `"auto"` — earthsearch in WASM, else NASA if a token is set, else MPC.
  Sentinel-1 radar always comes from Planetary Computer (anonymous).

- **`download_backend: "gee"` (optional).** If you *do* have a Google Earth
  Engine account, the compositing runs on Google's servers and you only
  download the result — less local CPU, but it needs
  `pip install earthengine-api` + `earthengine authenticate`. GEE is strictly
  optional; nothing about it is imported unless you opt in.

So the free path (NASA / Planetary Computer / AWS) is primary; GEE is a
convenience for those who already have it. There is a step-by-step guide for
the free tokens in the repo's `docs/manual_tokens_gratuitos.pdf`.


## What the browser cannot do — and why

Being honest about limits is part of using the tool well. The browser
(WebAssembly / Pyodide) is wonderful for *learning* and *small* jobs, but:

- **Memory.** Browser Python is **wasm32**: a hard ceiling around **4 GB**.
  A whole state or many-month cube does not fit; you would need tiling and
  streaming. The desktop tool has your machine's full RAM.
- **No numba, no GDAL binaries.** The fast desktop libraries (`pyshepseg`
  with numba, `earthengine-api`, TPOT's parallel search) either do not exist
  in Pyodide or run far slower. That is *why* `shepherd-wasm` exists — a
  pure-NumPy port so at least segmentation runs in the browser.
- **Compute time.** AutoML over hundreds of features across a region is
  minutes-to-hours of CPU — fine on a desktop, painful in a tab.

**The rule of thumb:** learn and prototype in the browser; run real regions
on the desktop (or the portable environment, next). Nothing you learned is
wasted — it is the *same pipeline*, just a bigger engine.


## Run the production pipeline yourself (offline, no GEE)

`geocrop_analysis_mx` installs with plain `pip` — **no conda, no Google Earth
Engine account** — and ships with the Yaqui Valley test data pre-processed,
so you can reproduce the whole thing offline. In a terminal (not this
browser):

```bash
git clone https://github.com/abxda/geocrop_analysis_mx
cd geocrop_analysis_mx
python -m venv .venv && . .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install -r requirements.txt

# Copy the bundled Yaqui test data into place (enables OFFLINE mode)
python src/main.py --config config.test.yaml --phase setup_test

# Run the full pipeline — it detects the offline mosaics and skips any
# download / GEE connection automatically:
python src/main.py --config config.test.yaml --phase full_run
```

You will watch the same seven phases you now understand — segment, label,
extract, train (TPOT), predict — end with a `predicted_map_test.gpkg` and a
`classification_report.txt` reporting about **89% accuracy**. Open the
GeoPackage in **QGIS** to explore your crop map as real, coordinate-aware
**vector** data.

**Optional — with GEE.** If you *do* have a Google Earth Engine account and
want to classify your own area for fresh dates, `config.yaml` shows how to
point at your own AOI and let the `download` phase build geomedians live.
GEE is an option, never a requirement.


## The portable environment (no browser, no install headaches)

There is a third way, between the browser and a full developer setup: a
**portable environment** — a self-contained Python that runs from a folder,
no admin rights, no conda. The sister project
**[portable-satelital](https://abxda.github.io/portable-satelital/)** already
solved this (a signed, relocatable Python plus a one-click launcher). The
same recipe applies here: distribute `geocrop_analysis_mx` with a portable
Python so a non-expert can double-click and run the desktop pipeline without
touching a terminal. Browser for learning, portable for real work on a
laptop, desktop/server for big regions — the same course, three engines.


## 🎓 You made it

You started not knowing what a pixel's reflectance was. You now understand —
and have *run* — every step from raw satellite light to a validated crop map,
and you know when to trust the browser, when to move to the desktop, and how
to read a classifier honestly. That is the whole point: not to click a
button, but to use `geocrop_analysis_mx` **consciously**, aware of its
concepts, its requirements, and its limits.

🔭 To keep going deeper on any concept, follow the **Go deeper** links in each
module — they open the bilingual concept cards of
**[rs-learning-audio](https://abxda.github.io/rs-learning-audio/)**, where
every idea has its prerequisites, lineage and references.


## 🔭 Go deeper

Optional: these bilingual concept cards expand what you just learned
(prerequisite chains, lineage to fundamentals, curated references):

- [Synthetic Aperture Radar (Sentinel-1)](https://abxda.github.io/rs-learning-audio/?id=synthetic-aperture-radar)
- [Radar Vegetation Index (RVI)](https://abxda.github.io/rs-learning-audio/?id=radar-vegetation-index)
- [The data cube (space × bands × time)](https://abxda.github.io/rs-learning-audio/?id=data-cube)
- [Big data in remote sensing](https://abxda.github.io/rs-learning-audio/?id=big-data)
- [Geospatial data & QGIS](https://abxda.github.io/rs-learning-audio/?id=geospatial-data)
- [Coordinate reference systems](https://abxda.github.io/rs-learning-audio/?id=coordinate-reference-system)
- [Raster data](https://abxda.github.io/rs-learning-audio/?id=raster)
- [Vector data (GeoPackage)](https://abxda.github.io/rs-learning-audio/?id=vector)



---

[← Previous · Module 8 — Capstone: the crop map, end to end](08_capstone_crop_map.ipynb)
